# Synaptic Tomogram Results Visualization

This notebook allows interactive exploration of analyzed synaptic tomograms, including overlays of membranes, vesicles, active zones, and AuNPs.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import json
import mrcfile
import ipywidgets as widgets
from IPython.display import display, clear_output

# Helper to find analyzed tomograms
def find_analyzed_tomograms(base_dir="../../data/"):
    tomos = []
    for root, dirs, files in os.walk(base_dir):
        if 'best_alignment' in dirs:
            tomo_path = Path(root)
            vesicle_json = tomo_path / 'best_alignment' / 'STT_results' / 'vesicles' / 'vesicle_results.json'
            if vesicle_json.exists():
                tomos.append(str(tomo_path))
    return sorted(tomos)

tomogram_paths = find_analyzed_tomograms()
tomo_selector = widgets.Dropdown(options=tomogram_paths, description='Tomogram:')
display(tomo_selector)

In [ ]:
def load_tomogram_slice(tomo_path, z_center=None):
    mrcs = list((Path(tomo_path) / 'best_alignment').glob('*ddw.mrc'))
    if not mrcs:
        return None
    with mrcfile.open(mrcs[0], 'r') as mrc:
        data = mrc.data
    if z_center is None:
        z_center = data.shape[0] // 2
    return data[z_center], z_center

def load_membrane_coords(tomo_path, kind='presynaptic'):
    aunps_dir = Path(tomo_path) / 'best_alignment' / 'aunps'
    files = sorted(aunps_dir.glob(f'{kind}membranes_*.txt'))
    coords = [np.loadtxt(f) for f in files if f.exists()]
    return coords

def load_active_zone_coords(tomo_path):
    az_dir = Path(tomo_path) / 'best_alignment' / 'STT_results' / 'active_zones'
    files = sorted(az_dir.glob('active_zone_pre*_post*_pre.txt'))
    coords = [np.loadtxt(f) for f in files if f.exists()]
    return coords

def load_vesicles(tomo_path):
    ves_file = Path(tomo_path) / 'best_alignment' / 'STT_results' / 'vesicles' / 'vesicle_results.json'
    with open(ves_file) as f:
        data = json.load(f)
    return data['vesicles']

def load_aunps(tomo_path):
    aunp_file = Path(tomo_path) / 'best_alignment' / 'aunps' / 'aunp_tm_BP_active_zone_all.star'
    try:
        import starfile
        star_data = starfile.read(aunp_file)
        if isinstance(star_data, dict):
            for v in star_data.values():
                if isinstance(v, pd.DataFrame):
                    return v
        elif isinstance(star_data, pd.DataFrame):
            return star_data
    except Exception as e:
        print(f'Could not load AuNPs: {e}')
    return None

In [ ]:
def plot_tomogram_overlays(tomo_path):
    vesicles = load_vesicles(tomo_path)
    pre_mem = load_membrane_coords(tomo_path, 'presynatptic')
    post_mem = load_membrane_coords(tomo_path, 'postsynaptic')
    azs = load_active_zone_coords(tomo_path)
    aunps = load_aunps(tomo_path)
    # Find z center of first active zone
    z_center = int(np.mean(azs[0][:,2])) if azs else None
    slice2d, zc = load_tomogram_slice(tomo_path, z_center)
    fig, ax = plt.subplots(figsize=(8,8))
    ax.imshow(slice2d, cmap='gray')
    # Overlay membranes
    for coords in pre_mem:
        ax.plot(coords[:,0], coords[:,1], 'r-', label='Presynaptic' if 'Presynaptic' not in ax.get_legend_handles_labels()[1] else '')
    for coords in post_mem:
        ax.plot(coords[:,0], coords[:,1], 'g-', label='Postsynaptic' if 'Postsynaptic' not in ax.get_legend_handles_labels()[1] else '')
    # Overlay vesicles
    for v in vesicles:
        c = np.array(v['center'])
        r = v['radius']
        circ = plt.Circle((c[0], c[1]), r, color='pink', fill=False, lw=1.5, label='Vesicle' if 'Vesicle' not in ax.get_legend_handles_labels()[1] else '')
        ax.add_patch(circ)
    # Highlight vesicles within 10 nm
    for v in vesicles:
        if v.get('distance_to_az', 99) <= 10:
            c = np.array(v['center'])
            r = v['radius']
            circ = plt.Circle((c[0], c[1]), r, color='aqua', fill=False, lw=2, label='<=10nm' if '<=10nm' not in ax.get_legend_handles_labels()[1] else '')
            ax.add_patch(circ)
    # Overlay active zone
    for coords in azs:
        ax.plot(coords[:,0], coords[:,1], 'y-', lw=2, label='Active Zone' if 'Active Zone' not in ax.get_legend_handles_labels()[1] else '')
    ax.legend()
    ax.set_title('Top-down slice with overlays')
    plt.show()

In [ ]:
def plot_3d_overlays(tomo_path):
    vesicles = load_vesicles(tomo_path)
    pre_mem = load_membrane_coords(tomo_path, 'presynatptic')
    post_mem = load_membrane_coords(tomo_path, 'postsynaptic')
    azs = load_active_zone_coords(tomo_path)
    aunps = load_aunps(tomo_path)
    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(111, projection='3d')
    # Membranes
    for coords in pre_mem:
        ax.plot(coords[:,0], coords[:,1], coords[:,2], 'r-', label='Presynaptic' if 'Presynaptic' not in ax.get_legend_handles_labels()[1] else '')
    for coords in post_mem:
        ax.plot(coords[:,0], coords[:,1], coords[:,2], 'g-', label='Postsynaptic' if 'Postsynaptic' not in ax.get_legend_handles_labels()[1] else '')
    # Vesicles
    for v in vesicles:
        c = np.array(v['center'])
        r = v['radius']
        u = np.linspace(0, 2*np.pi, 20)
        v_ = np.linspace(0, np.pi, 10)
        x = c[0] + r * np.outer(np.cos(u), np.sin(v_))
        y = c[1] + r * np.outer(np.sin(u), np.sin(v_))
        z = c[2] + r * np.outer(np.ones_like(u), np.cos(v_))
        ax.plot_wireframe(x, y, z, color='pink', alpha=0.5)
    # Highlight vesicles within 10 nm
    for v in vesicles:
        if v.get('distance_to_az', 99) <= 10:
            c = np.array(v['center'])
            r = v['radius']
            u = np.linspace(0, 2*np.pi, 20)
            v_ = np.linspace(0, np.pi, 10)
            x = c[0] + r * np.outer(np.cos(u), np.sin(v_))
            y = c[1] + r * np.outer(np.sin(u), np.sin(v_))
            z = c[2] + r * np.outer(np.ones_like(u), np.cos(v_))
            ax.plot_wireframe(x, y, z, color='aqua', alpha=0.7)
    # Active zones
    for coords in azs:
        ax.plot(coords[:,0], coords[:,1], coords[:,2], 'y-', lw=2, label='Active Zone' if 'Active Zone' not in ax.get_legend_handles_labels()[1] else '')
    # AuNPs
    if aunps is not None and not aunps.empty:
        ax.scatter(aunps['faCoordinateX'], aunps['faCoordinateY'], aunps['faCoordinateZ'], color='gold', s=40, label='AuNPs')
    ax.legend()
    ax.set_title('3D overlays')
    plt.show()

In [ ]:
def on_tomo_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        clear_output(wait=True)
        display(tomo_selector)
        plot_tomogram_overlays(change['new'])
        plot_3d_overlays(change['new'])

tomo_selector.observe(on_tomo_change)
if tomogram_paths:
    plot_tomogram_overlays(tomogram_paths[0])
    plot_3d_overlays(tomogram_paths[0])